# Model architecture
This notebook audits the exact frozen structure and creates the architecture figure from source.


In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve(); ROOT=ROOT.parent if ROOT.name=='notebooks' else ROOT
sys.path.insert(0,str(ROOT/'src'))
from phytospectra.model import PhytoCore, VisualTokenBridgeConfig, QwenAlignmentConfig
core=PhytoCore(); bridge=VisualTokenBridgeConfig(); qwen=QwenAlignmentConfig()
{'PhytoCore parameters':sum(p.numel() for p in core.parameters()), 'bridge':bridge, 'qwen':qwen}


## Data and model flow
1. The 204-band cube is masked and normalized using outer-training pixels only.
2. Spectral and spatial encoders create 50 and 132 tokens.
3. PhytoCore fuses 182 tokens and predicts three continuous and two categorical targets.
4. Direct token-zeroing sensitivity selects target-specific evidence; all 204 bands remain in the ROI.
5. Global memory, up to two local memories, target identity, and question embeddings enter the bridge.
6. The bridge emits 16 visual-prefix tokens of width 1536 for Qwen2.5-1.5B.
7. Matched LoRA/bridge alignment is followed by fixed two-pass class-aware remediation.
8. A fixed one-pass Cultivar sequence-ranking phase produces the R endpoint.
9. Q and R trainable tensors are interpolated with the frozen alpha=0.60; alpha is not selected per fold.
10. The frozen contract-optimization curriculum applies class balancing, token weighting, and inner-development-only selection.
11. Raw, constrained, decomposed-summary, and verifier-reranked outputs are reported separately.
12. Verified references resolve only to same-fold PhytoCore evidence.


In [ ]:
from phytospectra.visualization import architecture_figure
from IPython.display import Image, display
paths=architecture_figure(ROOT/'reports'/'architecture_only')
display(Image(filename=paths[0]))
